### Topic 6: Migrating the In-Memory Store into Qdrant

### Why Do We Need This Migration?

Earlier chapters stored document embeddings in a Python list and searched them using a brute-force loop with `np.dot()`. This approach is simple and works well for small datasets, but it has three limitations:

- Data is lost whenever the application restarts.
- Metadata filtering happens after retrieval, which may return fewer than the requested Top-K results.
- Every search compares the query with every stored vector, making search slower as the corpus grows.

Qdrant addresses these limitations by storing vectors inside a collection and using HNSW indexing for efficient retrieval.

### What Changes?

Only the **storage** and **retrieval** layers change.

**Before (In-Memory)**

- Embed all document chunks.
- Store embeddings in a Python list.
- Search using a `for` loop and `np.dot()`.
- Apply metadata filtering after retrieval.

**After (Qdrant)**

- Embed the same document chunks.
- Store embeddings as Points in a Qdrant collection.
- Search using `client.query_points()`.
- Apply metadata filtering during vector search.

### What Stays the Same?

The migration does **not** change the RAG pipeline.

The following remain identical:

- Embedding model
- Document chunks
- Corpus
- User query
- Retrieved results

Only the storage backend and search implementation change.

The purpose of this notebook is to demonstrate that an application can migrate from an in-memory vector store to Qdrant without changing the embedding pipeline or retrieval quality.

---

# End-to-End Flow of `qdrant_migration.py`

This example demonstrates how to migrate a vector store from an in-memory Python list to Qdrant while keeping the embedding pipeline unchanged.

The program performs the following steps in order:

1. Load the embedding model.
2. Create the Qdrant client.
3. Define the sample document chunks.
4. Build the in-memory vector store.
5. Perform semantic search using brute-force search.
6. Perform filtered search using post-search filtering.
7. Generate deterministic Point IDs.
8. Build the Qdrant store.
9. Perform semantic search using Qdrant.
10. Perform filtered search using Qdrant.
11. Compare both implementations.
12. Verify what changed and what stayed the same.

---

### Step 1: Load the Embedding Model

```python
model = SentenceTransformer(MODEL_NAME)
```

**What Happens:** Loads the Sentence Transformer model into memory. The same model is used by both the in-memory store and the Qdrant store.

**Theory:** The purpose of this notebook is to compare only the storage layer. Keeping the same embedding model ensures that any difference in retrieval comes from the storage implementation rather than the embeddings.

**Example:** The query **"What happens if I close my FD early?"** is embedded using the same model regardless of whether the vectors are stored in a Python list or in Qdrant.

**Internally:** The model loads its pre-trained weights and becomes ready to convert text into 384-dimensional embeddings.

---

### Step 2: Create the Qdrant Client

```python
client = QdrantClient(":memory:")
```

**What Happens:** Creates an in-memory Qdrant database for the migration demo.

**Theory:** Using `:memory:` removes the need for Docker while keeping the same Qdrant APIs. This allows the migration to be tested without changing any application logic.

**Example:** The same document chunks that were previously stored in a Python list will later be stored as Points inside this temporary Qdrant database.

**Internally:** An empty Qdrant instance is created. No collections or Points exist yet.

---

### Step 3: Define the Sample Corpus

```python
CHUNKS = [...]
```

**What Happens:** Creates the sample knowledge base used by both implementations.

**Theory:** Both the in-memory store and the Qdrant store must use exactly the same corpus to make the comparison fair. If the documents changed, retrieval differences could not be attributed solely to the storage layer.

**Example:** The corpus contains FD Policies, Product Guides, and FAQs, such as:

- Premature withdrawal policy.
- Senior citizen interest rates.
- FD FAQs.
- TDS policy.

**Internally:** The documents are stored as Python dictionaries. No embeddings are generated yet.

---

### Step 4: Build the In-Memory Vector Store

```python
inmemory_store = build_inmemory_store(CHUNKS)
```

Inside:

```python
texts = [c["text"] for c in chunks]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
)
```

**What Happens:** Converts every document into an embedding and stores it in a Python list.

**Theory:** This is the same implementation used in the earlier chapters. Each document is stored as a Python dictionary containing:

- `text`
- `source`
- `doc_type`
- `embedding`

Unlike Qdrant, the vectors exist only in application memory and disappear when the program exits.

**Example:** An FD FAQ is stored as:

```python
{
    "text": "...",
    "source": "FD_FAQ.json",
    "doc_type": "faq",
    "embedding": [...]
}
```

**Internally:** The embedding model generates one embedding for each document. Python stores these dictionaries inside a list without creating any index.

---

### Step 5: Perform In-Memory Semantic Search

```python
search_inmemory(
    inmemory_store,
    QUERY,
)
```

Inside:

```python
scores = [
    (
        np.dot(
            query_vector,
            entry["embedding"]
        ),
        entry
    )
    for entry in store
]
```

**What Happens:** Searches every stored document by comparing the query embedding with every document embedding.

**Theory:** This is a **brute-force search**. Since all embeddings are normalized, the dot product is equivalent to cosine similarity. Every document is compared, the scores are sorted, and the Top-K results are returned.

**Example:** If the query is:

> **"What happens if I close my FD early?"**

the program compares the query with all eight stored documents before selecting the three most similar ones.

**Internally:** Python loops through every entry in the list, computes one similarity score per document, sorts all scores, and returns the highest-scoring results.

---

### Step 6: Perform In-Memory Filtered Search

```python
search_inmemory_filtered(
    inmemory_store,
    QUERY,
    doc_type="faq",
)
```

**What Happens:** Performs a normal semantic search first, then removes documents whose `doc_type` does not match the filter.

**Theory:** Filtering happens **after** retrieval rather than during the search. This means some retrieved documents may be discarded, potentially leaving fewer than the requested Top-K matching results.

**Example:** Suppose the Top-3 results are:

- FAQ
- Policy
- Product

After filtering for `doc_type="faq"`, only the FAQ document remains, even though other FAQ documents exist further down the ranking.

**Internally:** The program performs the complete brute-force search, sorts every result, and then filters the returned documents based on the `doc_type` field.

---

### Step 7: Generate Deterministic Point IDs

```python
make_point_id(
    source,
    chunk_index,
)
```

**What Happens:** Generates a unique and deterministic ID for every document chunk before storing it in Qdrant.

**Theory:** Every Point in Qdrant requires a unique ID. A deterministic ID ensures that the same document always receives the same ID, allowing future upserts to update the existing Point instead of creating duplicates.

**Example:** The document:

- **Source:** `FD_Policy.json`
- **Chunk Index:** `0`

always generates the same Point ID, even if the document is re-ingested later.

**Internally:** The source name and chunk index are combined into a string, hashed using MD5, and converted into an integer.

---

### Step 8: Build the Qdrant Store

```python
build_qdrant_store(CHUNKS)
```

**What Happens:** Migrates the documents from the Python list into a Qdrant collection.

**Theory:** The embedding process remains exactly the same as the in-memory implementation. The only difference is how the embeddings are stored.

Instead of storing:

```python
{
    "text": ...,
    "embedding": ...
}
```

inside a Python list, each document becomes a Qdrant Point containing:

- ID
- Vector
- Payload

**Example:** An FD FAQ is stored as:

- **ID:** Deterministic Point ID
- **Vector:** 384-dimensional embedding
- **Payload:**
  - `text`
  - `source`
  - `doc_type`

**Internally:** Qdrant:

- Creates the collection.
- Generates embeddings.
- Converts every document into a Point.
- Stores all Points using `upsert()`.
- Builds the HNSW index automatically while inserting the vectors.

---

### Step 9: Perform Qdrant Semantic Search

```python
search_qdrant(
    QUERY,
)
```

Inside:

```python
client.query_points(...)
```

**What Happens:** Searches the Qdrant collection for the most semantically similar documents.

**Theory:** Unlike the in-memory implementation, Qdrant does not compare the query with every stored vector. It uses the HNSW index to efficiently locate the nearest neighbors.

**Example:** The query:

> **"What happens if I close my FD early?"**

returns the same relevant FD documents as the in-memory implementation, but the search is now handled by Qdrant instead of Python.

**Internally:** Qdrant:

- Receives the query embedding.
- Traverses the HNSW graph.
- Computes similarity scores for the most promising candidates.
- Returns the Top-K matching Points.

---

### Step 10: Perform Qdrant Filtered Search

```python
search_qdrant_filtered(
    QUERY,
    doc_type="faq",
)
```

**What Happens:** Searches only FAQ documents by applying the metadata filter during vector search.

**Theory:** Unlike the in-memory implementation, filtering is performed during retrieval rather than after retrieval. This makes the search more efficient and ensures that the returned results already satisfy the filter.

**Example:** When searching for:

```python
doc_type = "faq"
```

Qdrant searches only FAQ Points instead of searching every document and discarding non-FAQ results afterward.

**Internally:** Qdrant combines the HNSW search with the payload filter, so only matching Points participate in the nearest-neighbor search.

---

### Step 11: Compare Both Implementations

```python
print("WHAT CHANGED:")
print("WHAT STAYED THE SAME:")
```

**What Happens:** Compares the in-memory implementation with the Qdrant implementation to highlight exactly what changed during the migration.

**Theory:** A successful migration should modify only the storage and retrieval layers. The embedding pipeline, document structure, corpus, and retrieval quality should remain unchanged.

**What Changed**

- Storage moved from a Python list to a Qdrant collection.
- Manual `for` loop with `np.dot()` was replaced by `client.query_points()`.
- Metadata filtering moved from post-search filtering to filtering during vector search.

**What Stayed the Same**

- Embedding model.
- Document chunks.
- Corpus.
- User query.
- Retrieved results.

**Internally:** Both implementations use the same document embeddings. The only difference is that Qdrant manages vector storage, indexing, and retrieval internally instead of relying on Python loops.

---

### Understanding the Code Output

**TensorFlow Warning**

```text
The name tf.losses.sparse_softmax_cross_entropy is deprecated...
```

- This warning comes from TensorFlow/Keras, not from Qdrant.
- It is triggered internally by the Sentence Transformer library.
- It does not affect embedding generation or search results.

---

**In-Memory Store**

```text
In-memory store: 8 entries
```

- Confirms that all 8 document chunks were embedded and stored in a Python list.

---

**In-Memory Semantic Search**

```text
score=0.5106
score=0.4037
score=0.3879
```

- The first result has the highest similarity because it discusses premature FD withdrawal, matching the user's query.
- The second result is also related to premature closure but focuses on the penalty exception.
- The third result discusses Fixed Deposit interest rates, making it relevant but less similar.

---

**In-Memory Filtered Search**

```text
score=0.5106
score=0.3675
```

- Only FAQ documents remain after filtering.
- The filtering happens after the search, so non-FAQ documents are removed from the results.

---

**Qdrant Store**

```text
Qdrant store: 8 points
```

- Confirms that the same 8 documents were migrated into a Qdrant collection as Points.

---

**Qdrant Semantic Search**

```text
score=0.5106
score=0.4037
score=0.3879
```

- The scores are identical to the in-memory implementation.
- This confirms that the migration changed only the storage layer, not the retrieval quality.

---

**Qdrant Filtered Search**

```text
score=0.5106
score=0.3675
```

- Qdrant applies the metadata filter during vector search.
- Only FAQ Points participate in retrieval, making the search more efficient.

---

**Migration Summary**

```text
WHAT CHANGED:
```

- **Storage:** Python list → Qdrant collection.
- **Search:** `for` loop with `np.dot()` → `client.query_points()`.
- **Filtering:** Post-search filtering → Filtering during vector search.

```text
WHAT STAYED THE SAME:
```

- Embedding model.
- Document chunks.
- Corpus.
- User query.
- Retrieved results.

This confirms that the migration replaces only the storage and retrieval implementation while preserving the behavior of the application.

---

### Understanding Payload Filtering vs. Retrieval Strategy

To understand payload filtering, first look at how documents are stored in Qdrant.

During ingestion, every document is converted into a **Point** containing:

- ID
- Vector (Embedding)
- Payload (Metadata)

For example, the collection may contain:

**Point 1**

```python
payload = {
    "text": "Premature withdrawal incurs a 1 percent penalty.",
    "source": "FD_Policy.json",
    "doc_type": "policy"
}
```

**Point 2**

```python
payload = {
    "text": "Can I withdraw my FD before maturity?",
    "source": "FD_FAQ.json",
    "doc_type": "faq"
}
```

**Point 3**

```python
payload = {
    "text": "Senior citizens receive an additional 0.5 percent interest.",
    "source": "FD_Product_Guide.json",
    "doc_type": "product"
}
```

Notice that **`doc_type` is stored inside the payload** during `upsert()`. It is **not generated from the user's query**.

Later, during retrieval, the application sends:

```python
query_filter = Filter(
    must=[
        FieldCondition(
            key="doc_type",
            match=MatchValue(value="faq")
        )
    ]
)
```

This tells Qdrant:

> Search only those Points whose payload contains:

```python
doc_type = "faq"
```

Suppose the user asks:

```text
What is the FD rate?
```

The correct answer actually exists in a Policy document:

```text
The FD interest rate for 24 months is 7.1%.
```

However, because the filter is:

```python
doc_type = "faq"
```

Qdrant ignores all **Policy** and **Product** documents before semantic search begins.

The search is therefore performed only on FAQ Points, such as:

- Can I withdraw my FD before maturity?
- What is the minimum amount to open an FD?

Qdrant returns the closest FAQ because those are the only eligible documents, even though the Policy document is the best semantic match.

This naturally raises another question:

> Can the system first search all documents, determine that the best match belongs to the Policy category, and then search only Policy documents?

Yes. This is called a **two-stage retrieval strategy**, but it is implemented by the application, not by Qdrant.

A typical workflow is:

1. Search all documents without any filter.
2. Identify the top matching document.
3. Read its payload (for example, `doc_type="policy"`).
4. Perform a second search using `query_filter` with `doc_type="policy"`.
5. Return the retrieved Policy documents to the LLM.

This approach helps gather additional context from the same document category.

### Why Doesn't Qdrant Do This Automatically?

Qdrant is a **Vector Database**, not a retrieval orchestrator.

Its responsibility is to:

- Store vectors and payloads.
- Build and maintain the HNSW index.
- Perform semantic similarity search.
- Apply payload filters.
- Return the matching Points.

Qdrant simply executes the request it receives.

The application or RAG pipeline decides:

- Whether to search all documents or a specific category.
- Whether to apply a payload filter.
- Whether to perform one search or multiple searches.
- Whether to infer the document category from the first search.
- Whether to rerank or refine the retrieved results.

### Key Takeaway

The **query** tells Qdrant **what information the user is looking for**.

The **payload filter** tells Qdrant **which stored documents are allowed to participate in the search**.

These are independent of each other.

Qdrant does not decide the retrieval strategy—it efficiently executes the retrieval strategy designed by the application.

---

In [1]:
"""
qdrant_migration.py
---------------------
Side-by-side: the old in-memory vector store vs. the new Qdrant store.

Shows exactly what changes in the migration and what stays the same:
  SAME  : embedding model, chunk shape, corpus, query
  CHANGE: where vectors are stored + how nearest-neighbor search is called

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers numpy
"""

import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384

# load model once -- shared by both the old store and the new Qdrant store
model = SentenceTransformer(MODEL_NAME)

# in-memory Qdrant client -- same behavior as Docker, no persistence on restart
client = QdrantClient(":memory:")

# sample chunks -- same shape as chunker.py produces
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.",
     "source": "FD_Policy.json", "doc_type": "policy"},
]


# ======================================================================
# OLD: in-memory store (what earlier chapters used)
# ======================================================================

def build_inmemory_store(chunks: list) -> list:
    """Embeds all chunks and stores them in a plain Python list.
    This is exactly what the earlier chapter's knowledge base was."""

    # embed all texts in one batch call
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # store each chunk as a dict with its embedding attached
    store = []
    for i, chunk in enumerate(chunks):
        store.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "doc_type": chunk["doc_type"],
            "embedding": embeddings[i],  # numpy array stored directly in memory
        })
    return store


def search_inmemory(store: list, query: str, k: int = 3) -> list:
    """Brute-force search: compute dot product against every entry, sort, return top-k.
    Correct at small scale, scales linearly -- gets slower as corpus grows."""

    # embed the query the same way chunks were embedded
    query_vector = model.encode(query, normalize_embeddings=True)

    # compute cosine similarity against every stored entry (normalized -> dot product)
    scores = [(np.dot(query_vector, entry["embedding"]), entry) for entry in store]

    # sort by score descending and return top-k
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]


def search_inmemory_filtered(store: list, query: str, doc_type: str, k: int = 3) -> list:
    """Filtered search the old way: search everything, then discard non-matching.
    Problem: if most results are the wrong doc_type, we get fewer than k useful results."""

    query_vector = model.encode(query, normalize_embeddings=True)
    scores = [(np.dot(query_vector, entry["embedding"]), entry) for entry in store]
    scores.sort(key=lambda x: x[0], reverse=True)

    # filter AFTER fetching -- may return fewer than k if most results don't match
    filtered = [(s, e) for s, e in scores if e["doc_type"] == doc_type]
    return filtered[:k]


# ======================================================================
# NEW: Qdrant store (the migration target)
# ======================================================================

def make_point_id(source: str, chunk_index: int) -> int:
    """Deterministic ID so re-upserts update in place, not duplicate."""
    key = f"{source}::{chunk_index}"
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def build_qdrant_store(chunks: list) -> None:
    """Embeds all chunks and upserts them into Qdrant.
    The embedding step is identical to the in-memory version."""

    # create the collection -- same as Topic 5
    existing = [c.name for c in client.get_collections().collections]
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # embed all texts in one batch -- identical to the in-memory version
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build points -- vector goes to Qdrant, text/source/doc_type go to payload
    points = [
        PointStruct(
            id=make_point_id(chunks[i]["source"], i),
            vector=embeddings[i].tolist(),  # numpy array -> list for serialization
            payload={
                "text": chunks[i]["text"],          # stored for self-contained retrieval
                "source": chunks[i]["source"],       # provenance metadata
                "doc_type": chunks[i]["doc_type"],   # used for filtering
            },
        )
        for i in range(len(chunks))
    ]

    # upsert = insert new or replace existing -- idempotent
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Qdrant store: {client.get_collection(COLLECTION_NAME).points_count} points")


def search_qdrant(query: str, k: int = 3) -> None:
    """Qdrant search -- replaces the manual dot-product loop entirely.
    HNSW handles the nearest-neighbor search internally."""

    # embed the query -- identical to the in-memory version
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # one call replaces the entire for loop + sort from the in-memory version
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,     # the embedded query vector
        limit=k,                # return the top-k most similar points
        with_payload=True,      # include text/source/doc_type in results
    ).points                    # unwrap the response to get the list

    print(f"\nQdrant search: '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


def search_qdrant_filtered(query: str, doc_type: str, k: int = 3) -> None:
    """Qdrant filtered search -- filter applied DURING HNSW traversal.
    Always returns k results that match the filter, not k results then discard."""

    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[FieldCondition(key="doc_type", match=MatchValue(value=doc_type))]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nQdrant filtered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ======================================================================
# Run both side by side so the difference is visible
# ======================================================================

QUERY = "What happens if I close my FD early?"

print("=" * 60)
print("OLD: IN-MEMORY STORE")
print("=" * 60)

# build the old store
inmemory_store = build_inmemory_store(CHUNKS)
print(f"In-memory store: {len(inmemory_store)} entries")

# unfiltered search -- old way
print(f"\nIn-memory search: '{QUERY}'")
for score, entry in search_inmemory(inmemory_store, QUERY, k=3):
    print(f"  score={score:.4f} | [{entry['doc_type']}] {entry['text'][:60]}")

# filtered search -- old way (post-search filter, may return fewer than k)
print(f"\nIn-memory filtered search (doc_type='faq'): '{QUERY}'")
for score, entry in search_inmemory_filtered(inmemory_store, QUERY, doc_type="faq", k=3):
    print(f"  score={score:.4f} | [{entry['doc_type']}] {entry['text'][:60]}")

print("\n" + "=" * 60)
print("NEW: QDRANT STORE")
print("=" * 60)

# build the Qdrant store -- same embedding step, different storage
build_qdrant_store(CHUNKS)

# unfiltered search -- new way
search_qdrant(QUERY, k=3)

# filtered search -- new way (filter during HNSW, always returns k)
search_qdrant_filtered(QUERY, doc_type="faq", k=3)

print("\n" + "=" * 60)
print("WHAT CHANGED:")
print("  storage  : Python list  ->  Qdrant collection")
print("  search   : for loop + np.dot  ->  client.query_points()")
print("  filtering: post-search discard  ->  during HNSW traversal")
print("WHAT STAYED THE SAME:")
print("  embedding model, chunk shape, corpus, query, results")
print("=" * 60)


OLD: IN-MEMORY STORE
In-memory store: 8 entries

In-memory search: 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.4037 | [policy] This penalty does not apply if the FD is closed due to death
  score=0.3879 | [product] The FD interest rate for 24 months is 7.1 percent per annum.

In-memory filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

NEW: QDRANT STORE
Qdrant store: 8 points

Qdrant search: 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.4037 | [policy] This penalty does not apply if the FD is closed due to death
  score=0.3879 | [product] The FD interest rate for 24 months is 7.1 percent per annum.

Qdrant filtered search (d